# Tiled Attention (Flash Attention algorithm) — Exercise

Companion to [Attention Part 4 § The Full Algorithm](https://tanulsingh.github.io/blog/flash-attention#the-full-algorithm).

Now we combine online softmax with tiled QK^T and PV matmuls. The output is bit-equivalent to standard attention, but conceptually we never need to materialize the full N×N score matrix.

**Note**: in pure Python this won't be faster than standard attention — the speedup comes from staying in SRAM in a CUDA kernel. But the *algorithm* is what we're learning here.

## Setup

In [ ]:
# TODO: imports — torch, math

## TODO 1 — `flash_attention_v2`

Implement the FA2 outer-Q, inner-KV pattern from the blog:

```
for each block i of Q:
    Load Q_i
    Initialize running state: O_i = 0, m_i = -inf, l_i = 0
    
    for each block j of K, V:
        S_ij = Q_i @ K_j.T / sqrt(d)
        m_new = max(m_i, rowmax(S_ij))
        correction = exp(m_i - m_new)
        l_i = l_i * correction + rowsum(exp(S_ij - m_new))
        O_i = O_i * correction.unsqueeze(-1) + exp(S_ij - m_new) @ V_j
        m_i = m_new
    
    O_i = O_i / l_i.unsqueeze(-1)        # final normalization
```

In [ ]:
def flash_attention(Q, K, V, block_size: int = 64):
    """
    Args:
        Q: (n, d)
        K: (n, d)
        V: (n, d)
        block_size: how big each Q tile and KV tile is
    Returns:
        O: (n, d) — should be bit-equivalent to standard scaled dot-product attention
    """
    # TODO 1: implement the FA2 outer-Q inner-KV pattern
    pass

In [ ]:
# Sanity check:
#   - Q, K, V random of shape (256, 64)
#   - O_flash = flash_attention(Q, K, V, block_size=64)
#   - O_ref   = standard scaled_dot_product_attention(Q, K, V)
#   - Assert torch.allclose(O_flash, O_ref, atol=1e-5)
#   - Try varying block_size — all should match the reference

## Run the tests

In [ ]:
from tests import run_flash_attention_tests
# run_flash_attention_tests(flash_attention, flash_attention_causal)